In [1]:
from __future__ import annotations

import sys
from copy import deepcopy
from pathlib import Path

import torch
import yaml
from torch.utils.data import DataLoader

In [2]:
def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "configs" / "config_domainnet.yaml").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Could not find configs/config_domainnet.yaml")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT.parent))

from ebm_unlearning.src.data.domainnet import DomainNetSubset
from ebm_unlearning.src.data.split import ForgetSpec, RetainSpec, split_forget_retain, train_holdout_split
from ebm_unlearning.src.data.dataset import IndexedSubset
from ebm_unlearning.src.losses.clip_subspace import (
    compute_domainnet_subspace_weights,
    load_dino_encoder,
    WeightedSubset,
)
from ebm_unlearning.src.models.ebm import EnergyModel
from ebm_unlearning.src.training.pretrain import load_pretrained
from ebm_unlearning.src.training.unlearn import UnlearnConfig, unlearn
from ebm_unlearning.src.utils.logging import setup_logger
from ebm_unlearning.src.utils.seed import set_seed
from ebm_unlearning.src.utils.tracking import make_tracker

with open(ROOT / "configs" / "config_domainnet.yaml", "r") as f:
    cfg = yaml.safe_load(f)

set_seed(int(cfg["seed"]))
device = torch.device(cfg.get("device", "cpu"))

from datetime import datetime
logger  = setup_logger("unlearn", log_file=str(ROOT / "outputs" / "logs" / "unlearn_domainnet.log"))
run_id  = datetime.now().strftime("%Y%m%d-%H%M%S")
tracker = make_tracker(
    "tensorboard",
    log_dir=str(ROOT / "outputs" / "tensorboard" / "domainnet" / "unlearn" / run_id),
)

# Load full DomainNet subset (10 classes × 4 domains)
dset = DomainNetSubset(
    root=str(ROOT / cfg["data"]["data_dir"]),
    classes=cfg["data"]["classes"],
    domains=cfg["data"]["domains"],
)

# Forget: sketch-tiger only | Retain: everything else (incl. real/clipart/painting tiger)
forget_spec = ForgetSpec(
    mode="class_domain",
    class_label=int(cfg["data"]["forget"]["class_label"]),
    domain=str(cfg["data"]["forget"]["domain"]),
)
retain_spec = RetainSpec()
forget_all, retain_all = split_forget_retain(dset, forget_spec, retain_spec)

holdout_fraction = float(cfg["evaluation"]["holdout_fraction"])
forget_train, forget_holdout = train_holdout_split(forget_all, holdout_fraction, seed=int(cfg["seed"]))
retain_train, retain_holdout = train_holdout_split(retain_all, holdout_fraction, seed=int(cfg["seed"]) + 1)

batch_size  = int(cfg["data"]["batch_size"])
num_workers = int(cfg["data"]["num_workers"])

forget_loader = DataLoader(forget_train, batch_size=batch_size, shuffle=True,  num_workers=0, drop_last=True)
# retain_loader rebuilt below after DINO weights are computed

forget_name = cfg["data"]["forget"]["class_name"]
forget_domain = cfg["data"]["forget"]["domain"]
print(f"Forget : {forget_name} ({forget_domain}) — {len(forget_train)} train samples")
print(f"Retain : {len(retain_train)} train samples (includes real/clipart/painting {forget_name})")

2026-06-02 02:48:07.521276: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-02 02:48:07.565665: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-02 02:48:08.516973: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


[domainnet] 13262 images | 10 classes × 4 domains
Forget : tiger (sketch) — 309 train samples
Retain : 10301 train samples (includes real/clipart/painting tiger)


In [3]:
# ── CHANGE THESE only when encoder / forget class / k changes ─────────────────
ENCODER_BACKEND = "dino"   # "clip" | "dino"
# ─────────────────────────────────────────────────────────────────────────────

# Load E0 (pretrained reference — frozen)
E0 = EnergyModel(
    in_channels=int(cfg["model"]["in_channels"]),
    hidden_dim=int(cfg["model"]["hidden_dim"]),
    num_classes=int(cfg["model"].get("num_classes", 10)),
    embed_dim=int(cfg["model"].get("embed_dim", 128)),
    backbone=str(cfg["model"].get("backbone", "resnet18")),
    finetune_stages=int(cfg["model"].get("finetune_stages", 2)),
    imagenet_pretrained=bool(cfg["model"].get("imagenet_pretrained", True)),
)
E0 = load_pretrained(E0, str(ROOT / cfg["pretrain"]["checkpoint_path"]), device=device)

# Load DINOv2 encoder (frozen — only for subspace weight computation)
print(f"Loading {ENCODER_BACKEND.upper()} encoder...")
enc_model, enc_preprocess, enc_type = load_dino_encoder(device)
print(f"{ENCODER_BACKEND.upper()} loaded.")

# Build DINO-preprocessed loaders for feature extraction
from ebm_unlearning.src.data.domainnet import DomainNetSubset
dset_dino = DomainNetSubset(
    root=str(ROOT / cfg["data"]["data_dir"]),
    classes=cfg["data"]["classes"],
    domains=cfg["data"]["domains"],
    transform=enc_preprocess,   # DINOv2 preprocessing
)

from ebm_unlearning.src.data.split import split_forget_retain as sfr
forget_dino_all, retain_dino_all = sfr(dset_dino, forget_spec, retain_spec)
forget_dino_train, _ = train_holdout_split(forget_dino_all, holdout_fraction, seed=int(cfg["seed"]))
retain_dino_train, _ = train_holdout_split(retain_dino_all, holdout_fraction, seed=int(cfg["seed"]) + 1)

forget_dino_loader = DataLoader(forget_dino_train, batch_size=batch_size, shuffle=False, num_workers=0)
retain_dino_loader = DataLoader(retain_dino_train, batch_size=batch_size, shuffle=False, num_workers=0)

n_components = int(cfg["unlearning"].get("n_pca_components", 20))
print(f"\nComputing DINOv2 PCA subspace weights (k={n_components})...")
print(f"Forget subspace computed from: {forget_name} ({forget_domain}) only")

_retain_weights_cache, _, _ = compute_domainnet_subspace_weights(
    model=enc_model,
    forget_loader=forget_dino_loader,
    retain_loader=retain_dino_loader,
    device=device,
    n_components=n_components,
    encoder_type=enc_type,
)

# Zero out CLIP weights for non-forget-class retain samples.
# DINOv2 captures visual style as well as semantics: clipart images (stylized)
# project onto the tiger/sketch subspace due to style similarity, not because
# they are tigers. Applying a forget signal to bear/clipart or lion/clipart
# degrades their accuracy. Only tiger images in other domains should receive
# the cross-domain forget signal.
_forget_cls = int(cfg["data"]["forget"]["class_label"])
_retain_cls = retain_dino_train.base.targets[retain_dino_train.indices]
_retain_weights_cache = _retain_weights_cache.clone()
_retain_weights_cache[_retain_cls != _forget_cls] = 0.0
_n_tiger = int((_retain_cls == _forget_cls).sum())
print(f"  masked to {forget_name}-only: {_n_tiger} same-class retain samples retain non-zero weight")
print("Weights cached. Run next cell to configure lambda and train.")

Loading DINO encoder...


/home/owais/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/owais/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/owais/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


DINO loaded.
[domainnet] 13262 images | 10 classes × 4 domains

Computing DINOv2 PCA subspace weights (k=20)...
Forget subspace computed from: tiger (sketch) only
  DINO: extracting forget features (309 samples)...
  DINO: extracting retain features (10301 samples)...
  weights — mean=0.2199  max=0.9616  %>0.05: 94.2%
Weights cached. Run next cell to configure lambda and train.


In [11]:
# ── Run this cell whenever lambda_clip / steps changes ────────────────────────
with open(ROOT / "configs" / "config_domainnet.yaml") as f:
    cfg = yaml.safe_load(f)

checkpoint_path = str(ROOT / cfg["unlearning"]["checkpoint_path"])

un_cfg = UnlearnConfig(
    steps=int(cfg["unlearning"]["steps"]),
    lr=float(cfg["unlearning"]["lr"]),
    weight_decay=float(cfg["unlearning"]["weight_decay"]),
    lambda_f=float(cfg["unlearning"]["lambda_f"]),
    lambda_r=float(cfg["unlearning"]["lambda_r"]),
    lambda_m=float(cfg["unlearning"]["lambda_m"]),
    lambda_e=float(cfg["unlearning"]["lambda_e"]),
    lambda_clip=float(cfg["unlearning"].get("lambda_clip", 0.0)),
    n_pca_components=int(cfg["unlearning"].get("n_pca_components", 20)),
    margin=float(cfg["unlearning"]["margin"]),
    log_every=int(cfg["unlearning"]["log_every"]),
    checkpoint_path=checkpoint_path,
)

# Reset E to fresh trainable copy
E = deepcopy(E0)
E.train()
for p in E.parameters():
    p.requires_grad_(True)
if hasattr(E, '_backbone') and E._backbone is not None:
    E._set_resnet_trainable_stages(int(cfg["model"].get("finetune_stages", 2)))  # trainable backbone
   # E._set_resnet_trainable_stages(0)  # freeze backbone experiment — made forgetting worse

# Rebuild retain loader with DINO weights
retain_train_weighted = WeightedSubset(retain_train, _retain_weights_cache)
retain_loader = DataLoader(retain_train_weighted, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True)

trainable = sum(p.numel() for p in E.parameters() if p.requires_grad)
print(f"lambda_clip={un_cfg.lambda_clip}  steps={un_cfg.steps}  k={un_cfg.n_pca_components}")
print(f"Trainable params: {trainable:,}")
print(f"Checkpoint → {checkpoint_path}")

lambda_clip=0.5  steps=500  k=10
Trainable params: 10,560,513
Checkpoint → /home/owais/machine unlearning/ebm_unlearning/outputs/checkpoints/ebm_unlearned_domainnet_tiger_sketch.pt


In [12]:
E = unlearn(
    E,
    E0,
    forget_loader,
    retain_loader,
    device=device,
    cfg=un_cfg,
    logger=logger,
    tracker=tracker,
    seed=int(cfg["seed"]),
)
tracker.close()

[2026-06-02 03:42:45,606] [INFO] [unlearn] step=0 gap_fw=-0.2935 total=17.565531 forget=5.682431 retain=0.895597 margin=1.694061 energy_reg=14.300287 clip=2.437529
[2026-06-02 03:43:28,666] [INFO] [unlearn] step=50 gap_fw=3.9278 total=12.779772 forget=1.578373 retain=0.944162 margin=0.243523 energy_reg=27.692453 clip=2.977133
[2026-06-02 03:44:10,279] [INFO] [unlearn] step=100 gap_fw=5.7120 total=4.713658 forget=0.468125 retain=0.291833 margin=0.023698 energy_reg=31.885630 clip=2.543245
[2026-06-02 03:44:55,405] [INFO] [unlearn] step=150 gap_fw=7.1327 total=3.611968 forget=0.134993 retain=0.242709 margin=0.014132 energy_reg=36.649124 clip=1.998200
[2026-06-02 03:45:37,556] [INFO] [unlearn] step=200 gap_fw=7.6246 total=2.650450 forget=0.000000 retain=0.162436 margin=0.024087 energy_reg=40.932346 clip=1.922139
[2026-06-02 03:46:23,082] [INFO] [unlearn] step=250 gap_fw=7.6241 total=4.191975 forget=0.026768 retain=0.284833 margin=0.007170 energy_reg=38.787392 clip=2.541847
[2026-06-02 03:4

In [13]:
# ── Evaluation: per-class per-domain accuracy ─────────────────────────────────
# Run this after training to see cross-domain and cross-class generalization

from ebm_unlearning.src.evaluation.classification import predict_argmin_energy
import numpy as np
from collections import defaultdict

DOMAINS_EVAL = ["real", "sketch", "clipart", "painting"]
CLASS_NAMES  = dset.classes   # alphabetically sorted

def eval_per_domain_class(model, root, classes, domains, batch_size=32, num_classes=10):
    """Evaluate classification accuracy per (class, domain) pair."""
    results = {}
    for domain in domains:
        dset_d = DomainNetSubset(root=root, classes=classes, domains=[domain])
        loader_d = DataLoader(dset_d, batch_size=batch_size, shuffle=False, num_workers=0)
        y_true, y_pred = predict_argmin_energy(model, loader_d, device=device, num_classes=num_classes, y_chunk=5)
        for cls_idx, cls_name in enumerate(dset_d.classes):
            mask = y_true == cls_idx
            if mask.sum() == 0:
                continue
            acc = float(np.mean(y_pred[mask] == cls_idx))
            results[(cls_name, domain)] = acc
    return results

dn_root = str(ROOT / cfg["data"]["data_dir"])
classes = cfg["data"]["classes"]

print("Evaluating pretrained model...")
E0.eval()
pre_results = eval_per_domain_class(E0, dn_root, classes, DOMAINS_EVAL)

print("Evaluating unlearned model...")
E.eval()
unl_results = eval_per_domain_class(E,  dn_root, classes, DOMAINS_EVAL)

# Print comparison table
print(f"\n{'Class':12} {'Domain':10} {'Pretrained':>12} {'Unlearned':>10} {'Drop':>8}")
print("-" * 58)
focus_classes = ["tiger", "lion", "bear", "zebra", "dog", "truck", "guitar"]
for cls in focus_classes:
    for domain in DOMAINS_EVAL:
        key = (cls, domain)
        pre = pre_results.get(key, float("nan"))
        unl = unl_results.get(key, float("nan"))
        drop = pre - unl
        flag = " ← FORGET" if cls == forget_name and domain == forget_domain else ""
        if cls == forget_name or drop > 0.05:
            print(f"{cls:12} {domain:10} {pre:>11.1%} {unl:>10.1%} {drop:>+7.1%}{flag}")

Evaluating pretrained model...
[domainnet] 5905 images | 10 classes × 1 domains
[domainnet] 2510 images | 10 classes × 1 domains
[domainnet] 1383 images | 10 classes × 1 domains
[domainnet] 3464 images | 10 classes × 1 domains
Evaluating unlearned model...
[domainnet] 5905 images | 10 classes × 1 domains
[domainnet] 2510 images | 10 classes × 1 domains
[domainnet] 1383 images | 10 classes × 1 domains
[domainnet] 3464 images | 10 classes × 1 domains

Class        Domain       Pretrained  Unlearned     Drop
----------------------------------------------------------
tiger        real             98.4%      58.0%  +40.4%
tiger        sketch           98.7%      51.0%  +47.7% ← FORGET
tiger        clipart          99.4%      70.2%  +29.2%
tiger        painting         99.5%      55.9%  +43.6%
lion         real             94.4%      60.5%  +33.9%
lion         sketch           89.4%      63.0%  +26.4%
lion         clipart          82.6%      39.1%  +43.5%
lion         painting         92.9% 